# 01 · Data download and preprocessing

This notebook says where each dataset comes from and under which conditions it may be used, downloads the open datasets, and turns the raw files
into the tables and arrays that the other notebooks read.

| Output (paths in notebook 00) | Content |
|---|---|
| `MIMIC_RECORDS` | one row per MIMIC-IV-ECG record (row number = `idx`): patient, recording time, machine, machine measurements, rhythm |
| `MIMIC_REPORT_FLAGS` | whether the machine report mentions non-standard or additional leads, poor recording quality or lead reversal |
| `MIMIC_WAVE500`, `MIMIC_QC` | the 12-lead waveforms and the quality checks of every record |
| `CODE15_WAVE500`, `CODE15_EXAMS` | the CODE-15% waveforms in the same format, and the exam (age, sex, patient) behind each row |
| `PTBXL_WAVE500`, `PTBXL_RECORDS` | the PTB-XL waveforms in the same format, and the record (age, sex, patient) behind each row |
| `MIMIC_HUBERT`, `CODE15_HUBERT`, `PTBXL_HUBERT` | the inputs of HuBERT-ECG, which differ from the waveform format used by the other models |

All waveform arrays share one format (see notebook 00, *Data conventions*): 12 leads × 5,000 samples (10 s at 500 Hz), int16, 200 units per mV,
leads in the order of MIMIC-IV-ECG. Row `i` of a MIMIC-IV-ECG array is record `idx = i`, the `i`-th line of `record_list.csv`.

**Time and disk space.** On one desktop computer (24-core CPU, data on a hard disk), reading MIMIC-IV-ECG takes about 1 hour, converting CODE-15% about
45 minutes, PTB-XL a few minutes, and the HuBERT-ECG inputs about 45 minutes. The raw data take about 170 GB (plus about 85 GB of ZIP files during the
download, which can be deleted afterwards), and the outputs of this notebook about 170 GB. Every long step skips what exists and resumes after an
interruption.

**Credentialed data.** MIMIC-IV and MIMIC-IV-ED are only downloaded here; they are first used in notebook 02. MIMIC-IV-ECG, CODE-15% and PTB-XL are
open datasets.

In [ ]:
%run ./00_setup_and_settings.ipynb

## What to run

The download switches are off: the cells that download data run only when their switch is set to `True`.
`SUBSET` is for a functional check: with an integer `n`, every long step processes only `n` records spread evenly over each dataset and writes
its arrays to `SUBSET_DIR` (with the list of rows it processed), so that the full outputs are not touched. The tables (`MIMIC_RECORDS`,
`MIMIC_REPORT_FLAGS`, `CODE15_EXAMS`, `PTBXL_RECORDS`) take little time and are always made for every record.

In [ ]:
import hashlib
import json
import re
import shutil
import time
import urllib.request
import zipfile
from collections import deque
from concurrent.futures import ThreadPoolExecutor

import h5py
import pyarrow.parquet as pq
import wfdb
from scipy import signal as ss

DOWNLOAD_MIMIC_ECG = False      # MIMIC-IV-ECG 1.0, open (36 GB ZIP file)
DOWNLOAD_MIMIC_IV = False       # MIMIC-IV 3.1, credentialed: prints the commands to run in a terminal
DOWNLOAD_MIMIC_ED = False       # MIMIC-IV-ED 2.2, credentialed: prints the commands to run in a terminal
DOWNLOAD_CODE15 = False         # CODE-15% 1.0.0, open (46 GB of ZIP files)
DOWNLOAD_PTBXL = False          # PTB-XL 1.0.3, open (1.8 GB ZIP file)
VERIFY_CHECKSUMS = True         # check the small raw files used below against the checksums published with the datasets (a few seconds)

SUBSET = None                   # None: every record. An integer n: n records spread evenly over each dataset (functional check)
N_THREADS = 8                   # parallel threads of the long steps
SUBSET_DIR = DERIVED / f"subset_{SUBSET}"                       # arrays of a functional check (used only when SUBSET is set)
WORK = (DERIVED if SUBSET is None else SUBSET_DIR) / "_work"    # files of unfinished long steps (can be deleted when the notebook has finished)
WORK.mkdir(parents=True, exist_ok=True)


def output(path):
    """Where a long step writes: the path of notebook 00, or the subset folder during a functional check."""
    return path if SUBSET is None else SUBSET_DIR / path.name


def subset_rows(n):
    """Rows processed by the long steps: all, or SUBSET rows spread evenly from the first to the last."""
    return np.arange(n) if SUBSET is None else np.unique(np.linspace(0, n - 1, SUBSET).astype(np.int64))

## Helpers for downloading and checking files

In [ ]:
def file_digest(path, algorithm="sha256"):
    h = hashlib.new(algorithm)
    with open(path, "rb") as f:
        while chunk := f.read(1 << 22):
            h.update(chunk)
    return h.hexdigest()


def fetch(url, dest, size=None, md5=None):
    """Download `url` to `dest`. An interrupted download is resumed; the size and, where published, the MD5 checksum are checked."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and (size is None or dest.stat().st_size == size) and (md5 is None or file_digest(dest, "md5") == md5):
        print("present:", dest.name)
        return dest
    part = Path(str(dest) + ".part")
    have = part.stat().st_size if part.exists() else 0
    headers = {"User-Agent": "python-urllib"}
    if have:
        headers["Range"] = f"bytes={have}-"
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers)) as r, \
            open(part, "ab" if have and r.status == 206 else "wb") as f:
        shutil.copyfileobj(r, f, 1 << 22)
    if size is not None and part.stat().st_size != size:
        raise IOError(f"{dest.name}: incomplete download; run the cell again to resume")
    if md5 is not None and file_digest(part, "md5") != md5:
        raise IOError(f"{dest.name}: MD5 checksum differs from the published one; delete {part.name} and download again")
    part.replace(dest)
    print("downloaded:", dest.name)
    return dest


def extract_zip(zip_path, dest, strip_top=False):
    """Extract a ZIP file into `dest` (strip_top drops the top folder of the archive); files already extracted are skipped."""
    dest = Path(dest).resolve()
    with zipfile.ZipFile(zip_path) as z:
        for m in z.infolist():
            name = m.filename.split("/", 1)[1] if strip_top and "/" in m.filename else m.filename
            if m.is_dir() or not name:
                continue
            out = (dest / name).resolve()
            if dest not in out.parents:
                raise ValueError(f"{zip_path}: path outside the target folder")
            if out.exists() and out.stat().st_size == m.file_size:
                continue
            out.parent.mkdir(parents=True, exist_ok=True)
            with z.open(m) as src, open(out, "wb") as dst:
                shutil.copyfileobj(src, dst, 1 << 20)


def read_checksums(root):
    """{relative path: SHA-256} from the file SHA256SUMS.txt that PhysioNet publishes with each dataset."""
    sums = {}
    with open(Path(root) / "SHA256SUMS.txt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                digest, rel = line.split(maxsplit=1)
                sums[rel.strip().lstrip("*")] = digest.lower()
    return sums


def verify_sha256(root, names=None, sums=None):
    """Names of the files below `root` whose SHA-256 differs from SHA256SUMS.txt (all listed files, or only `names`)."""
    sums = read_checksums(root) if sums is None else sums
    names = list(sums) if names is None else names
    return [n for n in names if n not in sums or not (Path(root) / n).exists() or file_digest(Path(root) / n) != sums[n]]

## Datasets

Each dataset has its own folder below `RAW` (notebook 00). Read the terms of each dataset on its own page before you use it.

### MIMIC-IV-ECG 1.0

| | |
|---|---|
| Content | 800,035 diagnostic 12-lead ECGs (10 s, 500 Hz) of 161,352 patients, recorded in 2008–2019 at Beth Israel Deaconess Medical Center, with the measurements and statements of the recording machine |
| Source | PhysioNet, https://physionet.org/content/mimic-iv-ecg/1.0/ (MIMIC-IV-ECG: Diagnostic Electrocardiogram Matched Subset, version 1.0) |
| Licence and access | Open Data Commons Open Database License v1.0; open access, no account needed |
| Size | ZIP file 36 GB; 97 GB (1.6 million files) after extraction |
| Folder | `MIMIC_ECG_DIR`: `record_list.csv`, `machine_measurements.csv`, `SHA256SUMS.txt`, `files/p1000/…` |

The patient identifiers (`subject_id`) are those of MIMIC-IV, which links each ECG to the clinical records in notebook 02.
The cell downloads the ZIP file, extracts it and checks the tables against `SHA256SUMS.txt`; the waveform files are checked one by one when they are
read below. Extraction of 1.6 million files takes several hours on a hard disk.

In [ ]:
if DOWNLOAD_MIMIC_ECG:
    _zip = fetch("https://physionet.org/static/published-projects/mimic-iv-ecg/"
                 "mimic-iv-ecg-diagnostic-electrocardiogram-matched-subset-1.0.zip", RAW / "_downloads" / "mimic-iv-ecg-1.0.zip")
    extract_zip(_zip, MIMIC_ECG_DIR, strip_top=True)
    _sums = read_checksums(MIMIC_ECG_DIR)
    _bad = verify_sha256(MIMIC_ECG_DIR, [n for n in _sums if not n.endswith((".hea", ".dat"))], _sums)
    print("MIMIC-IV-ECG: tables and lists", "verified" if not _bad else f"FAILED: {_bad[:5]}")

### MIMIC-IV 3.1 (credentialed)

| | |
|---|---|
| Content | hospital and intensive care records of the patients of Beth Israel Deaconess Medical Center, 2008–2022; used here: patients (sex, anchor age and year, date of death), hospital transfers with the care unit, and stays in intensive care units |
| Source | PhysioNet, https://physionet.org/content/mimiciv/3.1/ |
| Licence and access | PhysioNet Credentialed Health Data License 1.5.0. Access requires a credentialed PhysioNet account, completion of the required training in human research (CITI "Data or Specimens Only Research"), and signing the data use agreement of the project on its page |
| Size | 10.7 GB in total, of which only three files are used here (next row) |
| Folder | `MIMIC_IV_DIR`: `hosp/patients.csv.gz`, `hosp/transfers.csv.gz`, `icu/icustays.csv.gz`, `SHA256SUMS.txt` |

PhysioNet asks for the password of your own account, so the download is run in a terminal, not in the notebook. Go to the folder `MIMIC_IV_DIR` and run

```
wget -N -c -x -nH --cut-dirs=3 --user <your PhysioNet username> --ask-password \
    https://physionet.org/files/mimiciv/3.1/SHA256SUMS.txt \
    https://physionet.org/files/mimiciv/3.1/hosp/patients.csv.gz \
    https://physionet.org/files/mimiciv/3.1/hosp/transfers.csv.gz \
    https://physionet.org/files/mimiciv/3.1/icu/icustays.csv.gz
```

(or `wget -r -N -c -np -nH --cut-dirs=3 --user <your PhysioNet username> --ask-password https://physionet.org/files/mimiciv/3.1/` for the whole
dataset). Never store the password in a file of this repository. With the switch on, the cell prints these commands for the folders of this computer.
The data use agreement forbids sharing the data and anything derived from them; this code writes such files only inside `MASTER`.

In [ ]:
def physionet_commands(folder, base, files):
    urls = " ".join(base + f for f in files)
    return f'cd "{folder}"\nwget -N -c -x -nH --cut-dirs=3 --user <your PhysioNet username> --ask-password {urls}'


if DOWNLOAD_MIMIC_IV:
    MIMIC_IV_DIR.mkdir(parents=True, exist_ok=True)
    print(physionet_commands(MIMIC_IV_DIR, "https://physionet.org/files/mimiciv/3.1/",
                             ["SHA256SUMS.txt", "hosp/patients.csv.gz", "hosp/transfers.csv.gz", "icu/icustays.csv.gz"]))

### MIMIC-IV-ED 2.2 (credentialed)

| | |
|---|---|
| Content | emergency department stays at Beth Israel Deaconess Medical Center, 2011–2019; used here: the stays (start and end) |
| Source | PhysioNet, https://physionet.org/content/mimic-iv-ed/2.2/ |
| Licence and access | as MIMIC-IV: PhysioNet Credentialed Health Data License 1.5.0, credentialed account, training, data use agreement signed on the page of the project |
| Size | 0.12 GB in total, of which only one file is used here (next row) |
| Folder | `MIMIC_ED_DIR`: `ed/edstays.csv.gz`, `SHA256SUMS.txt` |

In a terminal, in the folder `MIMIC_ED_DIR`:

```
wget -N -c -x -nH --cut-dirs=3 --user <your PhysioNet username> --ask-password \
    https://physionet.org/files/mimic-iv-ed/2.2/SHA256SUMS.txt \
    https://physionet.org/files/mimic-iv-ed/2.2/ed/edstays.csv.gz
```

In [ ]:
if DOWNLOAD_MIMIC_ED:
    MIMIC_ED_DIR.mkdir(parents=True, exist_ok=True)
    print(physionet_commands(MIMIC_ED_DIR, "https://physionet.org/files/mimic-iv-ed/2.2/",
                             ["SHA256SUMS.txt", "ed/edstays.csv.gz"]))

### CODE-15% 1.0.0

| | |
|---|---|
| Content | 345,779 12-lead ECG exams of 233,770 patients of the Telehealth Network of Minas Gerais, Brazil (2010–2016), a stratified 15% sample of the CODE study; 400 Hz, 7 or 10 s, stored as 4,096 samples with zero padding on both sides; age, sex and patient identifier of each exam |
| Source | Zenodo, https://zenodo.org/records/4916206 (doi:10.5281/zenodo.4916206) |
| Licence and access | Creative Commons Attribution 4.0 International; open access |
| Size | 46 GB of ZIP files; 68 GB of HDF5 files after extraction |
| Folder | `CODE15_DIR`: `exams.csv`, `exams_part0.hdf5` … `exams_part17.hdf5` |

The cell asks Zenodo for the list of files with their MD5 checksums, downloads each file (resuming interrupted downloads), checks it and extracts
the HDF5 file from each ZIP file. The ZIP files can be deleted afterwards.

In [ ]:
if DOWNLOAD_CODE15:
    with urllib.request.urlopen(urllib.request.Request("https://zenodo.org/api/records/4916206",
                                                       headers={"User-Agent": "python-urllib"})) as _r:
        _record = json.load(_r)
    for _f in sorted(_record["files"], key=lambda f: f["key"]):
        _dest = fetch(_f["links"]["self"], CODE15_DIR / _f["key"], size=_f["size"], md5=_f["checksum"].split(":", 1)[1])
        if _dest.suffix == ".zip" and not (CODE15_DIR / _dest.name.replace(".zip", ".hdf5")).exists():
            extract_zip(_dest, CODE15_DIR)
    print("CODE-15%:", len(list(CODE15_DIR.glob("exams_part*.hdf5"))), "HDF5 files")

### PTB-XL 1.0.3

| | |
|---|---|
| Content | 21,799 clinical 12-lead ECGs (10 s, 500 Hz and 100 Hz) of 18,869 patients, Physikalisch-Technische Bundesanstalt, Germany (1989–1996); age, sex and patient identifier of each record |
| Source | PhysioNet, https://physionet.org/content/ptb-xl/1.0.3/ |
| Licence and access | Creative Commons Attribution 4.0 International; open access |
| Size | ZIP file 1.8 GB; 3.2 GB (87,000 files) after extraction |
| Folder | `PTBXL_DIR`: `ptbxl_database.csv`, `records500/…`, `SHA256SUMS.txt` |

PTB-XL is used only to evaluate the models (notebook 04). The cell downloads the ZIP file, extracts it and checks every file against `SHA256SUMS.txt`.

In [ ]:
if DOWNLOAD_PTBXL:
    _zip = fetch("https://physionet.org/static/published-projects/ptb-xl/"
                 "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip", RAW / "_downloads" / "ptb-xl-1.0.3.zip")
    extract_zip(_zip, PTBXL_DIR, strip_top=True)
    _bad = verify_sha256(PTBXL_DIR)
    print("PTB-XL:", "all files verified" if not _bad else f"{len(_bad)} files FAILED, e.g. {_bad[:3]}")

### Check that the raw files are in place

Lists the files that the notebooks read and, with `VERIFY_CHECKSUMS`, compares the tables with the checksums published with the datasets.
The waveform files of MIMIC-IV-ECG are checked one by one while they are read (column `sha_ok` of `MIMIC_QC`).

In [ ]:
NEEDED = {
    "MIMIC-IV-ECG": (MIMIC_ECG_DIR, ["record_list.csv", "machine_measurements.csv", "SHA256SUMS.txt", "files"]),
    "MIMIC-IV": (MIMIC_IV_DIR, ["hosp/patients.csv.gz", "hosp/transfers.csv.gz", "icu/icustays.csv.gz"]),
    "MIMIC-IV-ED": (MIMIC_ED_DIR, ["ed/edstays.csv.gz"]),
    "CODE-15%": (CODE15_DIR, ["exams.csv"] + [f"exams_part{k}.hdf5" for k in range(18)]),
    "PTB-XL": (PTBXL_DIR, ["ptbxl_database.csv", "records500", "SHA256SUMS.txt"]),
}
for _name, (_root, _files) in NEEDED.items():
    _missing = [f for f in _files if not (_root / f).exists()]
    print(f"{_name:13s} {len(_files) - len(_missing)} of {len(_files)} present" + (f"; missing: {_missing}" if _missing else ""))
if VERIFY_CHECKSUMS:
    for _name, _root, _files in (("MIMIC-IV-ECG", MIMIC_ECG_DIR, ["record_list.csv", "machine_measurements.csv"]),
                                 ("MIMIC-IV", MIMIC_IV_DIR, NEEDED["MIMIC-IV"][1]),
                                 ("MIMIC-IV-ED", MIMIC_ED_DIR, NEEDED["MIMIC-IV-ED"][1]),
                                 ("PTB-XL", PTBXL_DIR, ["ptbxl_database.csv"])):
        if (_root / "SHA256SUMS.txt").exists():
            _bad = verify_sha256(_root, _files)
            print(f"{_name:13s} checksums of {len(_files)} files:", "OK" if not _bad else f"FAILED {_bad}")
        else:
            print(f"{_name:13s} no SHA256SUMS.txt; checksums not verified")

## Preprocessing

### A helper for the long steps

The long steps fill large arrays block by block. They write to a temporary file (`<name>.part`) and note the next block in a progress file, so that an
interrupted run continues where it stopped; the file gets its final name only when every block is written. A step whose outputs exist is skipped.

In [ ]:
def fill_arrays(targets, n, work, chunk, threads, label, checkpoint_seconds=60):
    """Fill the files of `targets` ({path: (shape of one row, dtype)}), each with n rows; work(lo, hi) returns {path: rows lo..hi-1}."""
    paths = list(targets)
    if all(p.exists() for p in paths):
        print(f"{label}: exists, kept")
        return
    if any(p.exists() for p in paths):
        raise FileExistsError(f"{label}: some outputs exist and others do not; move the existing ones away and run again")
    progress = WORK / (paths[0].name + ".progress.json")
    state = json.loads(progress.read_text()) if progress.exists() else {"next": 0, "complete": True}
    resume = not state.get("complete", False) and all((WORK / (p.name + ".part")).exists() for p in paths)   # an interrupted run of this step
    start = state["next"] if resume else 0
    mm = {p: np.lib.format.open_memmap(WORK / (p.name + ".part"), mode="r+" if resume else "w+", dtype=dt, shape=(n, *shape))
          for p, (shape, dt) in targets.items()}
    blocks = iter([(lo, min(lo + chunk, n)) for lo in range(start, n, chunk)])
    t0, done, last = time.time(), start, time.time()
    try:
        with ThreadPoolExecutor(threads) as ex:
            queue = deque((b, ex.submit(work, *b)) for b in [next(blocks, None) for _ in range(2 * threads)] if b)
            while queue:
                (lo, hi), fut = queue.popleft()
                for p, rows in fut.result().items():
                    mm[p][lo:hi] = rows
                nxt = next(blocks, None)
                if nxt:
                    queue.append((nxt, ex.submit(work, *nxt)))
                done = hi
                if time.time() - last > checkpoint_seconds or done == n:  # checkpoint about once a minute
                    for p in mm:
                        mm[p].flush()
                    progress.write_text(json.dumps({"next": done}))
                    rate = (done - start) / (time.time() - t0)
                    print(f"{label}: {done:,}/{n:,} rows ({rate:.0f}/s, about {(n - done) / max(rate, 1e-9) / 60:.0f} min left)", flush=True)
                    last = time.time()
    finally:                                                          # also after an interruption: close the files, so that the
        for p in list(mm):                                            # step can be resumed (and the files renamed) in the same session
            mm.pop(p).flush()
    for p in paths:
        (WORK / (p.name + ".part")).replace(p)
    progress.write_text(json.dumps({"next": n, "complete": True}))
    print(f"{label}: {n:,} rows written in {(time.time() - t0) / 60:.1f} min")


def to_int16(mv):
    """Millivolts -> int16 at WAVE_UNITS_PER_MV units per mV (rounded, clipped to the int16 range without the missing value)."""
    return np.clip(np.round(mv * float(WAVE_UNITS_PER_MV)), -32767, 32767).astype(np.int16)

### MIMIC-IV-ECG: records and machine measurements

One row per record, in the order of `record_list.csv` (`idx` = row). The recording time comes from `record_list.csv`, the machine (`cart_id`)
and its measurements from `machine_measurements.csv`. The measurements are kept only within plausible ranges, because the machine writes 29999
or more for a value it could not measure and occasionally gives implausible values:

| Column | Definition | Kept if |
|---|---|---|
| `hr` | 60,000 / RR interval (ms) | RR 200–3,000 ms |
| `pr_ms` | QRS onset − P onset | 50–400 ms |
| `qtc_ms` | QT / √(RR / 1,000 ms) (Bazett), with QT = T end − QRS onset kept if 200–700 ms | 250–700 ms |

The rhythm is read from the statements of the machine report, in this order of precedence: atrial fibrillation or flutter; paced
("paced", "pacing", "pacemaker"); sinus (sinus rhythm, tachycardia, bradycardia or arrhythmia); otherwise other. These statements come from the
recording machine and are not confirmed by a physician.

In [ ]:
# About 1 minute.
rl = pd.read_csv(MIMIC_ECG_DIR / "record_list.csv")
rl["idx"] = np.arange(len(rl))
print(f"MIMIC-IV-ECG: {len(rl):,} records")


def mimic_reports(df):
    """The machine report of each record as one lower-case text: the statements report_0 ... report_17 joined by ' | '."""
    cols = [c for c in df.columns if re.fullmatch(r"report_\d+", c)]
    return df[cols].fillna("").astype(str).agg(" | ".join, axis=1).str.lower()


if MIMIC_RECORDS.exists() and MIMIC_REPORT_FLAGS.exists():
    print("MIMIC_RECORDS and MIMIC_REPORT_FLAGS exist, kept")
else:
    mm_ = pd.read_csv(MIMIC_ECG_DIR / "machine_measurements.csv", low_memory=False)
    rec = rl.merge(mm_.drop(columns=["ecg_time"]), on=["subject_id", "study_id"], how="left", validate="one_to_one")
    del mm_
    report = mimic_reports(rec)

In [ ]:
if MIMIC_RECORDS.exists():
    print("MIMIC_RECORDS exists, kept")
else:
    t = pd.to_datetime(rec["ecg_time"])
    rec["minute"] = (t.dt.hour * 60 + t.dt.minute).astype("Int16")          # minute of the day of the recording, 0-1439
    for c in ("rr_interval", "p_onset", "qrs_onset", "t_end"):
        v = pd.to_numeric(rec[c], errors="coerce").astype(float)
        rec[c] = v.where(v.abs() < 29999)                                     # 29999 and above: not measured
    rr = rec["rr_interval"].where(rec["rr_interval"].between(200, 3000))
    rec["hr"] = 60000.0 / rr
    rec["pr_ms"] = (rec["qrs_onset"] - rec["p_onset"]).where(lambda s: s.between(50, 400))
    qt = (rec["t_end"] - rec["qrs_onset"]).where(lambda s: s.between(200, 700))
    rec["qtc_ms"] = (qt / np.sqrt(rr / 1000.0)).where(lambda s: s.between(250, 700))
    af = report.str.contains(r"atrial fibrillation|atrial flutter", regex=True)
    paced = report.str.contains(r"\bpac(?:ed|ing|emaker)\b", regex=True)
    sinus = report.str.contains(r"sinus (?:rhythm|tachycardia|bradycardia|arrhythmia)", regex=True)
    rec["rhythm"] = np.select([af, paced, sinus], ["af_flutter", "paced", "sinus"], "other")
    cols = ["idx", "subject_id", "ecg_time", "minute", "cart_id", "hr", "pr_ms", "qtc_ms", "rhythm"]
    rec[cols].to_parquet(MIMIC_RECORDS, index=False)
    print(f"MIMIC_RECORDS: {len(rec):,} records; rhythm:", rec["rhythm"].value_counts().to_dict())

### Machine-report flags

Statements of the machine report that point to a non-standard or poor recording. They are used to restrict the pairs within 1 hour to "clean" ECGs
(notebook 05). Each flag is a search of the lower-case report text for these terms:

| Flag | Terms (regular expressions) |
|---|---|
| `alt_leads`: non-standard or additional leads | `right[- ]sided`, `v[3-6]r\b`, `posterior lead`, `\bv[789]\b`, `v7-v9`, `lead placement` |
| `quality`: recording quality | `artifact`, `noise`, `baseline wander`, `technically`, `poor quality`, `poor data`, `limited`, `suboptimal`, `interference`, `tremor` |
| `reversal`: lead reversal or misplacement | `reversal`, `reversed`, `misplac`, `lead switch`, `swapped` |

`any` is set when at least one flag is.

In [ ]:
REPORT_FLAG_TERMS = {
    "alt_leads": r"right[- ]sided|v[3-6]r\b|posterior lead|\bv[789]\b|v7-v9|lead placement",
    "quality": r"artifact|noise|baseline wander|technically|poor quality|poor data|limited|suboptimal|interference|tremor",
    "reversal": r"reversal|reversed|misplac|lead switch|swapped",
}
if MIMIC_REPORT_FLAGS.exists():
    print("MIMIC_REPORT_FLAGS exists, kept")
else:
    flags = pd.DataFrame({"idx": rec["idx"]})
    for k, pattern in REPORT_FLAG_TERMS.items():
        flags[k] = report.str.contains(pattern, regex=True)
    flags["any"] = flags[list(REPORT_FLAG_TERMS)].any(axis=1)
    flags.to_parquet(MIMIC_REPORT_FLAGS, index=False)
    for k in [*REPORT_FLAG_TERMS, "any"]:
        print(f"  {k:10s} {int(flags[k].sum()):,} records ({100 * flags[k].mean():.2f}%)")
if "rec" in globals():
    del rec, report

### MIMIC-IV-ECG: waveforms and quality checks

Every record is read once, in the order of `record_list.csv`. For each record:

* `sha_ok`: the SHA-256 checksums of its header (`.hea`) and signal (`.dat`) files equal those in `SHA256SUMS.txt`;
* `format_ok`: the header describes 12 signals of 5,000 samples at 500 Hz in the expected lead order and the signal file has the expected size;
  the 16-bit samples are then stored unchanged (200 units per mV); `rescaled` marks a record with another gain or baseline, which is converted to
  200 units per mV (none in version 1.0);
* `n_missing`, `max_lead_missing_frac`: missing samples (−32768) in total and the largest share of missing samples in one lead;
* `n_flat_leads`: leads whose range (maximum − minimum of the samples present) is below 0.02 mV.

A record that cannot be read keeps −32768 everywhere and fails `sha_ok` and `format_ok`. Notebook 02 turns these checks into exclusions.

In [ ]:
QC_FIELDS = {"sha_ok": np.int8, "format_ok": np.int8, "rescaled": np.int8, "n_missing": np.int32, "max_lead_missing_frac": np.float32,
             "n_flat_leads": np.int8}
QC_DTYPE = np.dtype(list(QC_FIELDS.items()))


def parse_header(text):
    """Number of signals, sampling rate, samples, gains, baselines and lead names of a WFDB header."""
    lines = [ln for ln in text.splitlines() if ln.strip() and not ln.startswith("#")]
    rec_line = lines[0].split()
    nsig, fs, nsamp = int(rec_line[1]), float(rec_line[2]), int(rec_line[3])
    gains, bases, leads = [], [], []
    for ln in lines[1:1 + nsig]:
        f = ln.split()
        g = f[2].split("/")[0]                                    # e.g. "200.0(0)/mV": gain 200, baseline 0
        gains.append(float(g.split("(")[0]))
        bases.append(int(g.split("(")[1].rstrip(")")) if "(" in g else 0)
        leads.append(f[-1])
    return nsig, fs, nsamp, gains, bases, leads


def mimic_record(hea, dat, sha_hea, sha_dat):
    """(12, 5000) int16 waveform and the quality checks of one record from the bytes of its two files (None if they could not be read)."""
    q = dict.fromkeys(QC_FIELDS, 0)
    arr = np.full((12, 5000), WAVE_MISSING, np.int16)
    if hea is not None:
        q["sha_ok"] = int(hashlib.sha256(hea).hexdigest() == sha_hea and hashlib.sha256(dat).hexdigest() == sha_dat)
        nsig, fs, nsamp, gains, bases, leads = parse_header(hea.decode())
        if nsig == 12 and fs == 500 and nsamp == 5000 and leads == WAVE_LEADS and len(dat) == 120000:
            q["format_ok"] = 1
            d = np.frombuffer(dat, dtype="<i2").reshape(5000, 12).T       # format 16: samples interleaved by lead
            if any(g != 200.0 for g in gains) or any(b != 0 for b in bases):
                q["rescaled"] = 1
                v = np.round((d.astype(np.float64) - np.array(bases)[:, None]) * 200.0 / np.array(gains)[:, None])
                d = np.where(d == WAVE_MISSING, WAVE_MISSING, np.clip(v, -32767, 32767)).astype(np.int16)
            arr = np.ascontiguousarray(d)
    miss = arr == WAVE_MISSING
    q["n_missing"] = int(miss.sum())
    q["max_lead_missing_frac"] = float(miss.mean(axis=1).max())
    x = np.where(miss, 0, arr).astype(np.float32) / 200.0
    ptp = np.array([np.ptp(x[k][~miss[k]]) if (~miss[k]).any() else 0.0 for k in range(12)])
    q["n_flat_leads"] = int((ptp < 0.02).sum())
    return arr, tuple(q[k] for k in QC_FIELDS)

In [ ]:
# Reads 97 GB: about 1 hour from a hard disk; SUBSET = 2,000 takes about a minute.
_qc_npy = WORK / (output(MIMIC_QC).stem + ".npy")
if output(MIMIC_QC).exists() and output(MIMIC_WAVE500).exists():
    print("MIMIC_WAVE500 and MIMIC_QC exist, kept")
else:
    sums = {p: h for p, h in read_checksums(MIMIC_ECG_DIR).items() if p.endswith((".hea", ".dat"))}
    paths = rl["path"].tolist()
    rows = subset_rows(len(rl))

    def read_block(lo, hi):
        waves = np.empty((hi - lo, 12, 5000), np.int16)
        qc = np.empty(hi - lo, QC_DTYPE)
        for j, i in enumerate(rows[lo:hi]):
            base = MIMIC_ECG_DIR / paths[i]
            try:
                hea, dat = base.with_suffix(".hea").read_bytes(), base.with_suffix(".dat").read_bytes()
            except OSError:
                hea = dat = None
            waves[j], qc[j] = mimic_record(hea, dat, sums.get(paths[i] + ".hea"), sums.get(paths[i] + ".dat"))
        return {output(MIMIC_WAVE500): waves, _qc_npy: qc}

    fill_arrays({output(MIMIC_WAVE500): ((12, 5000), np.int16), _qc_npy: ((), QC_DTYPE)}, len(rows), read_block,
                chunk=256, threads=N_THREADS, label="MIMIC-IV-ECG waveforms")
    _q = np.load(_qc_npy)
    qc_table = pd.DataFrame({"idx": rows.astype(np.int64), **{k: _q[k] for k in QC_FIELDS}})
    qc_table.to_parquet(output(MIMIC_QC), index=False)
    if SUBSET is not None:
        np.save(SUBSET_DIR / "mimic_rows.npy", rows)
    del sums
    print("quality checks:", {"records": len(qc_table), "checksum failed": int((qc_table.sha_ok == 0).sum()),
                              "format failed": int((qc_table.format_ok == 0).sum()), "rescaled": int(qc_table.rescaled.sum()),
                              "with missing samples": int((qc_table.n_missing > 0).sum()),
                              "with a flat lead": int((qc_table.n_flat_leads > 0).sum())})

### CODE-15%: waveforms and exams

The exams are stored at 400 Hz as 4,096 samples (10.24 s), with zero padding around the 7-s or 10-s recording. Their amplitudes are larger than those
of recordings in millivolts by the factor `CODE_AMPLITUDE_RATIO` = 1.796 (notebook 00). Each exam is resampled to 500 Hz (polyphase filter, factor
5/4, giving 5,120 samples), the central 5,000 samples (10 s) are kept, the amplitude is divided by the ratio to obtain millivolts, the leads are put
in the MIMIC-IV-ECG order and the values are stored at 200 units per mV. A 7-s exam keeps its zero padding at both ends. Every model trained or
evaluated on CODE-15% receives these arrays.

**Rows.** The arrays hold the first 345,779 rows of the 18 HDF5 files read in order (`exams_part0` … `exams_part17`), as many rows as `exams.csv`
lists; the all-zero padding rows among them (exam identifier 0) have no age and are excluded in notebook 02, together with the exams outside
18–88 years.

In [ ]:
ex = pd.read_csv(CODE15_DIR / "exams.csv")
parts = sorted(ex["trace_file"].unique(), key=lambda s: int(s.split("part")[1].split(".")[0]))
n_code15 = len(ex)
part_ids = []
for p in parts:
    with h5py.File(CODE15_DIR / p, "r") as f:
        part_ids.append(f["exam_id"][:])
part_start = np.r_[0, np.cumsum([len(a) for a in part_ids])]           # first row of each part in the concatenated files
exam_ids = np.concatenate(part_ids)[:n_code15]
print(f"CODE-15%: {len(ex):,} exams in exams.csv; {part_start[-1]:,} rows in {len(parts)} HDF5 files; "
      f"rows used {n_code15:,}, of which padding rows {int((exam_ids == 0).sum())}")

if CODE15_EXAMS.exists():
    print("CODE15_EXAMS exists, kept")
else:
    exams = pd.DataFrame({"row": np.arange(n_code15), "exam_id": exam_ids}).merge(
        ex[["exam_id", "patient_id", "age", "is_male"]], on="exam_id", how="left")    # padding rows: no patient, age or sex
    exams.to_parquet(CODE15_EXAMS, index=False)
    print(f"CODE15_EXAMS: {len(exams):,} rows, {exams.patient_id.nunique():,} patients")


def code15_tracings(rows):
    """Raw tracings (len(rows), 4096, 12) of the given rows (sorted) of the concatenated HDF5 files."""
    out = np.empty((len(rows), 4096, 12), np.float32)
    part = np.searchsorted(part_start, rows, side="right") - 1
    for k in np.unique(part):
        sel = np.nonzero(part == k)[0]
        off = rows[sel] - part_start[k]
        with h5py.File(CODE15_DIR / parts[k], "r") as f:
            tr = f["tracings"]
            out[sel] = tr[off[0]:off[-1] + 1] if off[-1] - off[0] + 1 == len(off) else tr[off]
    return out


def code15_to_wave(tracings):
    """(b, 4096, 12) CODE-15% tracings -> (b, 12, 5000) int16 in the common format."""
    x = np.asarray(tracings, dtype=np.float32).transpose(0, 2, 1)                       # (b, 12, 4096)
    x = ss.resample_poly(x, 5, 4, axis=-1)[..., 60:5060] / CODE_AMPLITUDE_RATIO        # 500 Hz, central 10 s, mV
    return to_int16(x[:, LEAD_SWAP])                                                    # standard -> MIMIC lead order

In [ ]:
# About 45 minutes for the 345,779 exams; SUBSET = 2,000 takes under a minute.
rows_code15 = subset_rows(n_code15)
fill_arrays({output(CODE15_WAVE500): ((12, 5000), np.int16)}, len(rows_code15),
            lambda lo, hi: {output(CODE15_WAVE500): code15_to_wave(code15_tracings(rows_code15[lo:hi]))},
            chunk=512, threads=N_THREADS, label="CODE-15% waveforms")
if SUBSET is not None:
    np.save(SUBSET_DIR / "code15_rows.npy", rows_code15)

### PTB-XL: waveforms and records

The 500-Hz records (`filename_hr`) are read with `wfdb`, which returns millivolts; missing values (none in version 1.0.3) would become 0. The leads are put
in the MIMIC-IV-ECG order. Ages of 89 years or more are not given exactly in PTB-XL (they are coded as 300), so ages from 89 years are set to missing;
`age_valid` marks the records at ages of 18–88 years, the ones used for evaluation. `is_male` is 1 for sex code 0 (male) in PTB-XL.

In [ ]:
db = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv")
if PTBXL_RECORDS.exists():
    print("PTBXL_RECORDS exists, kept")
else:
    ptb = pd.DataFrame({"row": np.arange(len(db)), "ecg_id": db["ecg_id"], "patient_id": db["patient_id"],
                        "age": db["age"].where(db["age"] < 89), "is_male": (db["sex"] == 0).astype(int)})
    ptb["age_valid"] = ptb["age"].between(AGE_MIN, AGE_MAX)
    ptb.to_parquet(PTBXL_RECORDS, index=False)
    print(f"PTBXL_RECORDS: {len(ptb):,} records, ages 18-88: {int(ptb.age_valid.sum()):,}")


def ptbxl_block(rows):
    out = np.empty((len(rows), 12, 5000), np.int16)
    for j, i in enumerate(rows):
        sig, _ = wfdb.rdsamp(str(PTBXL_DIR / db["filename_hr"].iloc[i]))           # (5000, 12) in mV, standard lead order
        out[j] = to_int16(np.nan_to_num(sig.T[LEAD_SWAP]))
    return out


# About 1-2 minutes for the 21,799 records.
rows_ptbxl = subset_rows(len(db))
fill_arrays({output(PTBXL_WAVE500): ((12, 5000), np.int16)}, len(rows_ptbxl),
            lambda lo, hi: {output(PTBXL_WAVE500): ptbxl_block(rows_ptbxl[lo:hi])},
            chunk=256, threads=N_THREADS, label="PTB-XL waveforms")
if SUBSET is not None:
    np.save(SUBSET_DIR / "ptbxl_rows.npy", rows_ptbxl)

### Inputs of HuBERT-ECG

HuBERT-ECG expects its own input format, as described by its developers and implemented in their code: each lead is band-pass filtered
(0.05–47 Hz, a finite impulse response filter with 151 coefficients and a Hamming window, applied forward and backward), scaled to the range −1 to 1 by
its minimum and maximum over the 10 s, the record is cut into two windows of 5 s, and the 12 leads of each window (standard order) are concatenated and
downsampled by 5 (`scipy.signal.decimate`), which gives 2 × 6,000 values per ECG at 100 Hz. Missing samples are set to 0 first.
The filter has the settings of the developers' code and is designed, as in their filter library, with `scipy.signal.firwin`.

In [ ]:
HUBERT_FIR = ss.firwin(151, [0.05, 47.0], pass_zero=False, fs=500)     # order int(0.3 * 500) = 150 -> 151 taps


def hubert_input(x_int16):
    """(b, 12, 5000) int16 in the common format -> (b, 2, 6000) float16 input of HuBERT-ECG."""
    x = np.where(x_int16 == WAVE_MISSING, 0, x_int16).astype(np.float64)[:, LEAD_SWAP] / WAVE_UNITS_PER_MV
    x = ss.filtfilt(HUBERT_FIR, np.array([1]), x, axis=-1)
    lo, hi = x.min(-1, keepdims=True), x.max(-1, keepdims=True)
    x = 2 * (x - lo) / (hi - lo + 1e-8) - 1
    out = np.empty((len(x), 2, 6000), np.float16)
    for k, s in enumerate((0, 2500)):
        out[:, k] = ss.decimate(x[..., s:s + 2500].reshape(len(x), -1), 5, axis=-1).astype(np.float16)
    return out


# About 45 minutes for the three datasets (about 450 records per second): MIMIC-IV-ECG 30, CODE-15% 13, PTB-XL 1.
for _src, _dst in ((MIMIC_WAVE500, MIMIC_HUBERT), (CODE15_WAVE500, CODE15_HUBERT), (PTBXL_WAVE500, PTBXL_HUBERT)):
    _wave = np.load(output(_src), mmap_mode="r")
    fill_arrays({output(_dst): ((2, 6000), np.float16)}, len(_wave),
                lambda lo, hi, w=_wave, d=output(_dst): {d: hubert_input(np.asarray(w[lo:hi]))},
                chunk=1024, threads=N_THREADS, label=f"HuBERT-ECG inputs from {_src.name}")
    del _wave

### Summary

In [ ]:
for _p in (MIMIC_RECORDS, MIMIC_REPORT_FLAGS, output(MIMIC_QC), CODE15_EXAMS, PTBXL_RECORDS):
    print(f"{_p.name:32s} {pq.ParquetFile(_p).metadata.num_rows:>9,} rows")
for _p in (MIMIC_WAVE500, MIMIC_HUBERT, CODE15_WAVE500, CODE15_HUBERT, PTBXL_WAVE500, PTBXL_HUBERT):
    _a = np.load(output(_p), mmap_mode="r")
    print(f"{output(_p).name:32s} {str(_a.shape):>20s} {_a.dtype}")
    del _a